# **매장·상품군 수요 패턴 군집**

전체 정확도만 보면 **어떤 수요에서 예측이 쓸모 있는지** 알기 어렵습니다. 매장×상품군 시계열을 판매 규모, 0 판매 비율, 변동성, 요일성 등으로 묶고, 각 유형의 특징을 설명합니다. 개별 상품이나 고객의 구매 순서를 분석하는 것은 아닙니다.

군집은 Valid 시작 전인 **2017-07-18까지의 180일**로 정의합니다. 이후 판매량은 군집 생성에 넣지 않으며, [08 평가](08_model_evaluation_and_decision.ipynb)에서 유형별 예측 오류를 읽는 데 사용합니다.

## **환경 준비**

별도의 00 노트북에서 저장소를 My Drive에 clone·갱신한 뒤 실행합니다. Drive를 연결하고 `/content/drive/MyDrive/retail-demand-forecasting`을 작업 폴더로 사용합니다.


In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount("/content/drive/")
except ImportError:
    pass

REPO = Path("/content/drive/MyDrive/retail-demand-forecasting")
REPO = REPO.parent if REPO.name == "notebooks" else REPO
assert (REPO / "pyproject.toml").is_file(), "저장소 루트에서 실행해 주세요."
os.chdir(REPO)

## **패키지 준비**

Colab에서는 먼저 별도의 00 노트북으로 Drive를 연결하고 저장소와 Favorita `train.csv`를 준비합니다. 로컬에서는 저장소 루트 또는 `notebooks` 폴더에서 실행할 수 있습니다. 기본 패키지는 `requirements.txt`에 포함되어 있습니다. UMAP 2차원 시각화에만 필요한 `umap-learn`은 이 노트북에서만 추가로 설치합니다.

Colab에서는 **환경 준비 → 패키지 설치 → Library** 셀 순서로 실행합니다. 런타임을 재시작했다면 설치 셀도 다시 실행해야 `retail_forecast`를 가져올 수 있습니다.

In [ ]:
%pip install -q pandas pyarrow scikit-learn seaborn koreanize-matplotlib
%pip install -q -e "{REPO}" --no-deps
%pip install -q umap-learn


## **Library**

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from importlib.util import find_spec
from matplotlib import font_manager

from sklearn.preprocessing import StandardScaler
from umap import UMAP

from retail_forecast.segments import (
    CLUSTER_COLUMNS,
    build_segment_features,
    fit_demand_segments,
    load_segment_history,
    summarize_segments,
)


## **Utils**

대용량 원본 CSV의 기간 선택과 군집 Feature 계산은 `src/retail_forecast/segments.py`에 있습니다. 이 노트북은 설정·시각화·해석만 담당합니다.

## **분석 설정**

In [ ]:
font_path = Path(find_spec("koreanize_matplotlib").origin).parent / "fonts" / "NanumGothic.ttf"
font_manager.fontManager.addfont(font_path)
sns.set_theme(style="whitegrid", font=font_manager.FontProperties(fname=font_path).get_name())
plt.rcParams["axes.unicode_minus"] = False

CUTOFF = pd.Timestamp("2017-07-18")
HISTORY_DAYS = 180
N_CLUSTERS = 5
MIN_COVERAGE = 0.90
UMAP_NEIGHBORS = 30
UMAP_MIN_DIST = 0.1

RAW_PATH = REPO / "data" / "raw"
OUTPUT_DIR = REPO / "outputs" / "demand_segments"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert (RAW_PATH / "train.csv").is_file(), RAW_PATH / "train.csv"

## **1. 군집에 사용할 과거 판매 이력**

- `train.csv`에서 **2017-07-18까지 180일**만 읽습니다. 0 판매는 실제 관측으로 남기고, 기록 자체가 없는 날은 `coverage_ratio`로 구분합니다.
- 기간·행·시계열 수는 입력 범위를 확인하는 표입니다. 아래 첫 5행은 **원본 관측**이며, 군집에는 이를 그대로 180개 열로 넣지 않고 다음 단계에서 시계열별로 요약합니다.

In [ ]:
history = load_segment_history(RAW_PATH, CUTOFF, HISTORY_DAYS)
history_overview = pd.DataFrame([{
    "시작일": history["date"].min().date(),
    "마지막일": history["date"].max().date(),
    "관측 행": len(history),
    "매장·상품군": history[["store_nbr", "family"]].drop_duplicates().shape[0],
    "0 판매 비율": history["sales"].eq(0).mean(),
}])
history_overview

### **군집 전 원본 관측 예시**

- 한 행은 특정 날짜의 매장×상품군 판매 기록입니다. 판촉 수량 `onpromotion`은 참고로 보존하지만, 군집 생성 입력으로 쓰지 않습니다.

In [ ]:
history_head = history[["date", "store_nbr", "family", "sales", "onpromotion"]].head()
history_head

## **2. 180일 기록을 시계열별 특성으로 요약**

매장×상품군 하나를 **시계열 한 개**로 봅니다. 아래 다섯 값만 표준화해 KMeans에 넣습니다.

| 입력 열 | 계산 방식 | 군집에서 구분하려는 수요 행동 |
|---|---|---|
| `log_mean_sales` | `log(1 + 일평균 판매량)` | 평소 판매 규모 |
| `zero_share` | 판매량이 0인 관측일 비율 | 간헐적·무판매 수요 |
| `log_cv` | 일별 판매량의 변동계수에 `log(1+x)` | 평소 대비 일별 변동 |
| `log_weekday_cv` | 요일별 평균 판매량의 변동계수에 `log(1+x)` | 요일별 수준 차이. **주간 반복성 자체는 아님** |
| `trend_log` | 마지막 28일과 처음 28일 평균의 `log(1+x)` 차이 | 최근 증가·감소 방향 |

`promo_day_share`는 판촉 상품이 있었던 날의 비율입니다. 판매 행동과 함께 해석하되 **군집 입력에서는 제외**합니다. `coverage_ratio`는 180일 중 기록이 있는 날의 비율이며, 90% 미만은 `이력 부족`으로 남깁니다. `trend_log`는 계산 후 ±2로 제한하므로 경계값에 모인 군집은 해석에 주의합니다.

### **군집 분리 정도와 이력 부족 여부**


In [ ]:
segment_features = build_segment_features(history, CUTOFF, HISTORY_DAYS)
feature_head = segment_features[[
    "store_nbr", "family", "observed_days", "mean_sales", "zero_share",
    "log_cv", "log_weekday_cv", "trend_log", "promo_day_share", "coverage_ratio",
]].head()
feature_head

### **군집에 실제로 들어가는 값**

- 위 표는 **원 단위의 관측값**과 집계값을 함께 보여줍니다. 아래 표의 `CLUSTER_COLUMNS` 다섯 열만 표준화한 후 KMeans에 입력합니다.
- 같은 열을 모든 매장×상품군 시계열에 적용하며, 미래 Valid·Test의 판매량은 포함하지 않습니다.

In [ ]:
cluster_input_head = segment_features.loc[
    segment_features["coverage_ratio"].ge(MIN_COVERAGE),
    ["store_nbr", "family", *CLUSTER_COLUMNS],
].head()
cluster_input_head

### **군집 수와 분리 정도**

- 같은 다섯 입력으로 3개·5개 군집을 비교하면 실루엣은 각각 약 **0.517·0.378**입니다. 3개는 판매량 약 **98.6%**를 한 군집에 묶어 운영상 차이를 보기 어렵습니다.
- 따라서 5개를 **해석용 구분**으로 사용합니다. 실루엣 최고값을 택한 것은 아니며, 작은 군집과 추세 경계값은 별도 확인합니다. 번호는 성능 순위가 아닙니다.

In [ ]:
segment_features, silhouette = fit_demand_segments(
    segment_features, clusters=N_CLUSTERS, min_coverage=MIN_COVERAGE,
)
profile = summarize_segments(segment_features)
segment_features["history_cutoff"] = CUTOFF
segment_features["cluster_schema"] = "demand_5_without_promo_v1"

segment_features.to_parquet(OUTPUT_DIR / "segment_features.parquet", index=False)
profile.to_csv(OUTPUT_DIR / "segment_profile.csv", index=False)
segment_quality = pd.DataFrame([{"Silhouette": silhouette, "이력 부족 시계열":
                       segment_features["segment"].eq("이력 부족").sum()}])
segment_quality

### **군집별 전형적인 판매 패턴**

- 각 군집의 중앙값은 **전형적인 시계열**을 설명합니다. 이 표만으로 공급 우선순위를 정하면 안 됩니다. 다음 표의 전체 판매량 비중도 함께 봅니다.
- `median_promo_share`는 군집을 만든 원인이 아니라, 군집을 만든 **뒤에 비교한 판촉 특성**입니다.

In [ ]:
profile_view = profile.round(3)
profile_view


### **군집의 크기와 실제 판매량 비중**

- `series_share_pct`는 매장×상품군 조합의 비중, `sales_share_pct`는 관측된 180일 **판매 수량의 비중**입니다.
- 담당자는 많은 시계열을 가진 군집보다 **판매 수량이 많이 몰린 군집**을 먼저 확인할 수 있습니다. 매출·이익 비중은 가격 정보가 없어 계산하지 않습니다.

In [ ]:
segment_volume = segment_features.assign(
    history_sales=segment_features["mean_sales"] * segment_features["observed_days"]
).groupby("segment", observed=True)["history_sales"].sum()
segment_overview = profile[["segment", "series", "median_daily_sales", "median_zero_share"]].copy()
segment_overview["series_share_pct"] = 100 * segment_overview["series"] / segment_overview["series"].sum()
segment_overview["sales_share_pct"] = 100 * segment_overview["segment"].map(segment_volume) / segment_volume.sum()
segment_overview.to_csv(OUTPUT_DIR / "segment_volume_overview.csv", index=False)
segment_volume_view = segment_overview.round(3)
segment_volume_view

### **시계열 개수와 판매량 비중의 차이**

- 두 막대가 크게 다르면 시계열 수만으로 업무 중요도를 판단할 수 없습니다. 특히 판매량 비중이 큰 군집은 예측 오류가 전사 수요 계획에 더 크게 반영됩니다.

In [ ]:
volume_chart = segment_overview.melt(
    id_vars="segment", value_vars=["series_share_pct", "sales_share_pct"],
    var_name="구분", value_name="비중",
)
volume_chart["구분"] = volume_chart["구분"].map({
    "series_share_pct": "시계열 수", "sales_share_pct": "판매 수량",
})
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=volume_chart, x="segment", y="비중", hue="구분", ax=ax)
ax.set(xlabel="군집", ylabel="전체에서 차지하는 비중 (%)", title="군집별 시계열 수와 판매 수량")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "segment_volume_share.png", dpi=160)
plt.show()

## **3. 군집을 점검하는 보조 시각화**

- 다음 산점도·Heatmap·UMAP은 군집이 어떤 입력값으로 나뉘었는지 살피는 **진단 자료**입니다. 운영 판단은 위의 판매량 비중과 [08의 군집별 예측 오류](08_model_evaluation_and_decision.ipynb)를 함께 봅니다.
- 산점도는 판매 규모와 0 판매 비율 두 축만 보여주므로, 다섯 입력 전체로 나눈 군집이 겹쳐 보여도 이상하지 않습니다.

In [ ]:
clustered = segment_features.loc[segment_features["segment"].ne("이력 부족")]
fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=clustered, x="log_mean_sales", y="zero_share", hue="segment",
                alpha=0.65, s=35, ax=ax)
ax.set(xlabel="log(1 + 일평균 판매량)", ylabel="0 판매 비율", title="매장·상품군 수요 유형")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "segment_scatter.png", dpi=160)
plt.show()

In [ ]:
feature_medians = clustered.groupby("segment", observed=True)[CLUSTER_COLUMNS].median()
feature_scale = clustered[CLUSTER_COLUMNS].std(ddof=0).replace(0, 1)
relative_profile = (feature_medians - clustered[CLUSTER_COLUMNS].mean()) / feature_scale

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(relative_profile.T, annot=True, fmt=".1f", cmap="vlag", center=0, ax=ax)
ax.set(xlabel="군집", ylabel="", title="전체 시계열 평균 대비 군집 특징 (표준편차 단위)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "segment_profile_heatmap.png", dpi=160)
plt.show()

### **UMAP으로 다섯 수요 특성을 2차원에서 점검**

- 군집을 다시 학습하지 않습니다. 동일한 다섯 입력을 2차원에 놓아 **가까운 시계열의 배치**를 살핍니다.
- 축의 숫자와 군집 사이 거리는 발주량이나 정확도를 뜻하지 않습니다. 중요한 결론은 위의 판매량 비중과 08의 오류 비교에서 읽습니다.

In [ ]:
umap_source = segment_features.loc[
    segment_features["segment"].ne("이력 부족"),
    ["segment", *CLUSTER_COLUMNS],
].copy()
scaled_features = StandardScaler().fit_transform(umap_source[CLUSTER_COLUMNS])
umap_model = UMAP(
    n_components=2, n_neighbors=UMAP_NEIGHBORS, min_dist=UMAP_MIN_DIST,
    random_state=42, n_jobs=1,
)
coordinates = umap_model.fit_transform(scaled_features)

umap_view = umap_source[["segment"]].copy()
umap_view["UMAP 1"] = coordinates[:, 0]
umap_view["UMAP 2"] = coordinates[:, 1]

fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(
    data=umap_view, x="UMAP 1", y="UMAP 2", hue="segment",
    alpha=0.55, s=28, ax=ax,
)
ax.set(title="매장·상품군 수요 군집 · UMAP 2차원")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "segment_umap.png", dpi=160)
plt.show()


## **결과를 읽는 순서**

1. 각 군집의 시계열 수와 판매 규모를 확인합니다. 작은 군집이나 `이력 부족`은 별도로 표시합니다.
2. 0 판매 비율·요일성·변동성·추세로 **패턴을 설명하는 이름**을 붙입니다. 번호 자체에는 의미가 없습니다.
3. [08 평가 노트북](08_model_evaluation_and_decision.ipynb)의 군집별 7일 WAPE·Bias·판매량 비중을 확인합니다. 02를 실행한 뒤 08의 분석 셀을 다시 실행하면 됩니다. 모델 학습은 필요하지 않습니다.

이 군집은 현재 한 시점의 기술적 요약입니다. 시점이 바뀌면 군집 경계와 소속도 바뀔 수 있으므로 운영 규칙으로 바로 고정하지 않습니다.